In [ ]:
#!pip install scikit-learn

In [ ]:
import numpy as np
from numpy import ndarray
from torchvision import datasets
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.model_selection import train_test_split



plt.style.use('dark_background')



In [ ]:
class Conv2d:
    def __init__(
            self,
            in_channels: int,
            out_channels: int,
            kernel_size: int,
            stride: int = 1,
            padding: int = 0,
            bias: bool = True
    ):
        """
        Инициализация свёрточного слоя.

        :param in_channels: количество входных каналов
        :param out_channels: количество выходных каналов (фильтров)
        :param kernel_size: размер ядра свёртки (квадратное)
        :param stride: шаг свёртки
        :param padding: размер отступа (zero-padding)
        :param bias: смещение
        """
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.bias = bias

        # Инициализация весов по методу He
        fan_in = in_channels * kernel_size * kernel_size
        self.w = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * np.sqrt(2.0 / fan_in)
        if self.bias:
            self.b = np.zeros((out_channels))
        else:
            self.b = None

        # Для backward pass
        self.dw = None
        self.db = None
        self.dx = None

    def conv(self, left_arr, right_arr, pad_override=None, stride_override=None, mode='forward'):
        """
        Универсальный метод свёртки.

        Поддерживает три режима:
        - 'forward': стандартная свёртка (N, C_in, H_in, W_in) * (C_out, C_in, K, K) -> (N, C_out, H_out, W_out)
        - 'dw': вычисление градиента по весам (N, C_in, H_in, W_in) ⊛ (N, C_out, H_out, W_out) -> (C_out, C_in, K, K)
        - 'dx': вычисление градиента по входу (N, C_out, H_out, W_out) * (C_in, C_out, K, K) -> (N, C_in, H_in, W_in)

        :param left_arr: массив справа от свёртки
        :param right_arr: веса или градиент в зависимости от mode
        :param pad_override: переопределение padding
        :param stride_override: переопределение stride
        :param mode: режим работы ('forward', 'dw', 'dx')
        :return: результат свёртки
        """
        pad = self.padding if pad_override is None else pad_override
        stride = self.stride if stride_override is None else stride_override

        if mode == 'forward':
            return self._conv_forward(left_arr, right_arr, pad, stride)
        elif mode == 'dw':
            return self._conv_grad_weight(left_arr, right_arr, pad, stride)
        elif mode == 'dx':
            return self._conv_grad_input(left_arr, right_arr, pad, stride)
        else:
            raise ValueError(f"Unknown mode: {mode}")

    def _conv_forward(self, x, w, pad, stride):
        """Стандартная свёртка для прямого прохода"""
        N, C_in, H, W = x.shape
        C_out, _, K, _ = w.shape

        # Размеры выхода
        H_out = (H + 2 * pad - K) // stride + 1
        W_out = (W + 2 * pad - K) // stride + 1

        # Padding
        if pad > 0:
            x_pad = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)), mode='constant')
        else:
            x_pad = x

        out = np.zeros((N, C_out, H_out, W_out))

        # Операция свёртки
        for i in range(H_out):
            for j in range(W_out):
                h_idx = slice(i * stride, i * stride + K)
                w_idx = slice(j * stride, j * stride + K)
                region = x_pad[:, np.newaxis, :, h_idx, w_idx]
                out[..., i, j] = np.sum(region * w, axis=(2, 3, 4))

        return out

    def _conv_grad_weight(self, x, dout, pad, stride):
        """
        Вычисление градиента по весам.
        """
        N, C_in, H, W = x.shape
        _, C_out, _, _ = dout.shape
        K = self.kernel_size

        H_out = H + 2*pad - K + 1
        W_out = W + 2*pad - K + 1

        dout_dil = np.zeros((N, C_out, H_out, W_out), dtype=dout.dtype)
        dout_dil[..., ::self.stride, ::self.stride] = dout
        dout = dout_dil[:, :, None, :, :]

        dw = np.zeros((C_out, C_in, K, K))

        # Padding для входа
        if pad > 0:
            x_pad = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)), mode='constant')
        else:
            x_pad = x

        # Операция свёртки
        for i in range(K):
            for j in range(K):
                h_idx = slice(i, i + H_out)
                w_idx = slice(j, j + W_out)
                region = x_pad[:, np.newaxis, :, h_idx, w_idx]
                dw[..., i, j] = np.sum(region * dout, axis=(0, 3, 4))

        return dw

    def _conv_grad_input(self, dout, w, pad, stride):
        """
        Вычисление градиента по входу.
        Используется для backward pass при вычислении dx.
        """
        N, _, H_in, W_in = self.x.shape
        C_in, C_out, K, _ = w.shape

        if stride < self.stride:
            H_out = H_in + K - 2*pad - 1
            W_out = W_in + K - 2*pad - 1
            dout_dil = np.zeros((N, C_out, H_out, W_out), dtype=dout.dtype)
            dout_dil[..., ::self.stride, ::self.stride] = dout
            dout = dout_dil

        N, C_out, H_out, W_out = dout.shape

        # Padding для входа
        if pad > 0:
            dout_pad = np.pad(dout, ((0, 0), (0, 0), (pad, pad), (pad, pad)), mode='constant')
        elif pad < 0:
            dout_pad = dout[..., -pad:pad, -pad:pad]
        else:
            dout_pad = dout

        dx = np.zeros((N, C_in, H_in, W_in))


        # Операция свёртки
        for i in range(H_in):
            for j in range(W_in):
                h_idx = slice(i, i + K)
                w_idx = slice(j, j + K)
                region = dout_pad[:, np.newaxis, :, h_idx, w_idx]
                dx[..., i, j] = np.sum(region * w, axis=(2, 3, 4))

        return dx

    def forward(self, x):
        """
        Прямой проход.
        :param x: вход, форма (N, C_in, H, W)
        :return: выход, форма (N, C_out, H_out, W_out)
        """

        self.x = x
        out = self.conv(self.x, self.w, mode='forward')

        if self.bias:
            out +=self.b[:, None, None]

        return out

    def backward(self, dout):
        """
        Обратный проход.
        :param dout: градиент функции потерь по выходу, форма (N, C_out, H_out, W_out)
        :return: dx: градиент по входу, форма (N, C_in, H, W)
        """
        N, C_in, H, W = self.x.shape

        # 1. Градиент по смещениям
        if self.bias:
            self.db = np.sum(dout, axis=(0, 2, 3), keepdims=False)

        # 2. Градиент по весам
        self.dw = self.conv(self.x, dout, stride_override=1, mode='dw')

        # 3. Градиент по входу
        # Поворот фильтров на 180° и транспонирование каналов
        w_rot = self.w[:, :, ::-1, ::-1]              # flip по высоте и ширине
        w_dx = w_rot.transpose(1, 0, 2, 3)            # C_in <-> C_out
        pad_dx = self.kernel_size - 1 - self.padding  # padding

        self.dx = self.conv(
            dout, w_dx,
            pad_override=pad_dx,
            stride_override=1,
            mode='dx'
        )

        return self.dx

### Реализация SGD + momentum.

In [ ]:
class SGD:
    """
    SGD optimizer + momentum
    """
    def __init__(self, lr: float = 1e-3, beta1: float = 0.9, weight_decay: float = 0):
        self.lr = lr
        self.beta1 = beta1
        self.weight_decay = weight_decay

        # Состояния оптимизатора.
        self.m = []

    def step(self, params: list[ndarray], grads: list[ndarray]) -> list[ndarray]:
        """
        Выполняет один шаг оптимизации.

        Args:
            params: list[ndarray] – текущие параметры модели
            grads:  list[ndarray] – градиенты для параметров

        Returns:
            list[np.ndarray] – обновлённые параметры
        """

        for i, (param, grad) in enumerate(zip(params, grads)):

            # Инициализация состояний при первом шаге
            if len(self.m) != len(params):
                self.m.append(np.zeros_like(param))

            m = self.m[i]

            # Обновление скользящего среднего
            m = self.beta1 * m + grad

            # SGD + momentum: шаг опритизации
            param_new = (
                param
                - self.lr * m
                - self.lr * self.weight_decay * param
            )

            # Сохраняем состояния и новые параметры
            self.m[i] = m
            params[i] = param_new

        return params



### Реализация RMSprop.

In [ ]:
class RMSprop:
    """
    RMSprop optimizer.
    """
    def __init__(
            self,
            lr: float = 1e-3,
            beta2: float = 0.999,
            eps: float = 1e-8,
            weight_decay: float = 0
    ):
        self.lr = lr
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay

        # Состояния оптимизатора.
        self.v = []

    def step(self, params: list[ndarray], grads: list[ndarray]) -> list[ndarray]:
        """
        Выполняет один шаг оптимизации.

        Args:
            params: list[np.ndarray] – текущие параметры модели
            grads:  list[np.ndarray] – градиенты для параметров

        Returns:
            list[np.ndarray] – обновлённые параметры
        """

        for i, (param, grad) in enumerate(zip(params, grads)):

            # Инициализация состояний при первом шаге
            if len(self.v) != len(params):
                self.v.append(np.zeros_like(param))

            v = self.v[i]

            # Обновление скользящего среднего
            v = self.beta2 * v + (1 - self.beta2) * np.square(grad)

            # RMSprop: шаг опритизации
            lr_step = self.lr / (np.sqrt(v) + self.eps)
            param_new = (
                param
                - lr_step * grad
                - self.lr * self.weight_decay * param
            )

            # Сохраняем состояния и новые параметры
            self.v[i] = v
            params[i] = param_new

        return params

### Реализация AdamW.

In [ ]:
class AdamW:
    """
    AdamW optimizer.
    """
    def __init__(
            self,
            lr: float = 1e-3,
            beta1: float = 0.9,
            beta2: float = 0.999,
            eps: float = 1e-8,
            weight_decay: float = 1e-2
    ):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.t = 0  # счётчик шагов

        # Состояния оптимизатора: первые и вторые моменты
        self.m = []
        self.v = []

    def step(self, params: list[ndarray], grads: list[ndarray]) -> list[ndarray]:
        """
        Выполняет один шаг оптимизации.

        Args:
            params: list[np.ndarray] – текущие параметры модели
            grads:  list[np.ndarray] – градиенты для параметров

        Returns:
            list[np.ndarray] – обновлённые параметры
        """
        self.t += 1

        for i, (param, grad) in enumerate(zip(params, grads)):

            # Инициализация состояний при первом шаге
            if len(self.v) != len(params):
                self.m.append(np.zeros_like(param))
                self.v.append(np.zeros_like(param))

            m = self.m[i]
            v = self.v[i]

            # Обновление скользящих средних
            m = self.beta1 * m + (1 - self.beta1) * grad
            v = self.beta2 * v + (1 - self.beta2) * np.square(grad)

            # Коррекция смещения
            m_hat = m / (1 - self.beta1 ** self.t)
            v_hat = v / (1 - self.beta2 ** self.t)

            # AdamW: шаг оптимизации
            adam_step = m_hat / (np.sqrt(v_hat) + self.eps)
            param_new = (
                param
                - self.lr * adam_step
                - self.lr * self.weight_decay * param
            )

            # Сохраняем состояния и новые параметры
            self.m[i] = m
            self.v[i] = v
            params[i] = param_new

        return params

### Реализация Dropout.

In [ ]:
class Dropout:
    """ Слой Dropout для регуляризации нейронной сети. """

    def __init__(self, p: float = 0.5, train: bool = True):

        self.p = p              # вероятность обнуления
        self.training = train   # режим: обучение / inference

        self.scale = 1.0 / (1.0 - self.p)   # для масштабирования

    def forward(self, x: ndarray) -> ndarray:
        # Во время inference
        if not self.training:
            return x

        # Генерирую маску для нейронов
        self.mask = (np.random.rand(*x.shape) > self.p)

        return x * self.mask * self.scale

    def backward(self, grad: ndarray) -> ndarray:
        return grad * self.mask * self.scale

    def __call__(self, x: ndarray) -> ndarray:
        return self.forward(x)

### Обучение сети.

In [ ]:
class CrossEntropyLoss:
    """
    Функция потерь Cross-Entropy для задач многоклассовой классификации.
    Реализация: Softmax + Cross-Entropy Loss.
    """

    def forward(self, pred: ndarray, target: ndarray) -> float:
        """
        Прямой проход через Cross-Entropy loss.

        pred   : выход с последнего слоя сети, shape (bs, C)
        target : one-hot метки классов, shape (bs, C)
        """
        self.batch_size = pred.shape[0]
        # Сохраняю target для backward.
        self.target = target

        # Вычитаю максимум из exp. для численно стабильных вычислений.
        logits_max = pred.max(axis=1, keepdims=True)
        exp_pred = np.exp(pred - logits_max)

        # Считаю и сохраняю Softmax для backward.
        self.softmax = exp_pred / exp_pred.sum(axis=1, keepdims=True)

        # log-sum-exp
        log_sum_exp = np.log(exp_pred.sum(axis=1, keepdims=True))

        # Считаю функцию потерь: Softmax + Cross-Entropy Loss.
        loss = (
            -np.sum(target * pred, axis=1, keepdims=True)
            + logits_max
            + log_sum_exp
        )

        return loss.mean()

    def backward(self) -> ndarray:
        """
        Обратный проход через: Softmax + Cross-Entropy Loss.
        """
        return (self.softmax - self.target) / self.batch_size

    def __call__(self, pred: ndarray, target: ndarray) -> float:
        return self.forward(pred, target)

# ReLU
def relu(x: ndarray) -> ndarray:
    return np.maximum(0, x)

def drelu(act: ndarray, grad: ndarray) -> ndarray:
    return grad*np.where(act>0, 1, 0)

# Softmax
def softmax(x: ndarray) -> ndarray:
    max_x = np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x - max_x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def load_mnist_simple(
    train: bool = True,
    root: str = "./data",
) -> tuple[ndarray, ndarray]:
    num_class = 10

    dataset = datasets.MNIST(
        root=root,
        train=train,
        download=True
    )

    images = np.stack([img for img, _ in dataset])      # (N, 28, 28)
    images = images.astype(np.uint8)

    labels = np.array([label for _, label in dataset])  # (N,)
    one_hot = np.eye(num_class)[labels]                 # (N,num_class)

    return images, one_hot

def label_to_onehot(label: int, num_class: int = 10) -> ndarray:
    one_hot = np.zeros(num_class)
    one_hot[label] = 1
    return one_hot

def onehot_to_label(one_hot: ndarray) -> ndarray:
    return np.argmax(one_hot)

def to_label(pred_bs: ndarray) -> ndarray:
    return np.argmax(pred_bs, axis=-1)

def normalize(sample: ndarray) -> ndarray:
    s_min = np.min(sample, axis=(2, 3, 4), keepdims=True)
    s_max = np.max(sample, axis=(2, 3, 4), keepdims=True)
    return (sample - s_min) / (s_max - s_min)*2 - 1

def initialization_param(
        size: tuple[int, int],
        mode: str = "randn",
        bound: float = None,
        seed: int = None
) -> ndarray:
    rng = np.random.default_rng(seed)

    if mode == "randn":
        return rng.standard_normal(size=size)
    elif mode == "uniform":
        return rng.uniform(-bound, bound, size=size)
    else:
        raise ValueError(f"Параметр 'mode' принимает два занчения: 'randn' и 'uniform'. Вы передали '{mode}'.")

def create_layers(
        in_features: int,
        out_features: int,
        hidden_size: int = 128,
        mode: str = "randn",
        seed: int = None
) -> tuple[ndarray, ndarray, ndarray, ndarray]:
    # Первый линейный слой
    W1 = initialization_param((in_features, hidden_size), mode=mode, bound=np.sqrt(6 / in_features), seed=seed)
    b1 = initialization_param((1, hidden_size), mode=mode, bound=np.sqrt(6 / hidden_size), seed=seed)

    # Второй линейный слой
    W2 = initialization_param((hidden_size, out_features), mode=mode, bound=np.sqrt(6 / hidden_size), seed=seed)
    b2 = initialization_param((1, out_features), mode=mode, bound=np.sqrt(6 / 2), seed=seed)

    return W1, b1, W2, b2

def data_loader(
        data: tuple[ndarray, ndarray],
        in_features: int,
        out_features: int,
        batch_size: int,
        epoch: int,
        shuffle: bool = False,
        seed: int = None,
):
    imgs, targets = data
    data_size = imgs.shape[0]
    num_batches = data_size // batch_size
    total = num_batches * batch_size
    in_channels = 1
    H_in = 28
    W_in = 28

    # Перемешивание данных
    if shuffle:
        rng = np.random.default_rng(seed + epoch if seed else None)
        idx = rng.permutation(data_size)
        imgs = imgs[idx]
        targets = targets[idx]

    # Формирование батчей
    imgs_batches = imgs[:total].reshape(num_batches, batch_size, in_channels, H_in, W_in)
    targets_batches = targets[:total].reshape(num_batches, batch_size, out_features)

    return imgs_batches, targets_batches

def train(
    loss_model,
    opt,
    data: tuple,    # (train_data, val_data)
    in_features: int,
    hidden_size: int,
    out_features: int,
    init_mode: str = "randn",
    batch_size: int = 16,
    epochs: int = 1,
    shuffle: bool = False,
    norm: bool = False,
    seed: int | None = None,
) -> tuple[ndarray, ndarray, ndarray, ndarray]:

    train_data = data['train']
    val_data = data['val']

    # Создаю параметры нейронной сети.
    conv1 = Conv2d(
        in_channels = 1,
        out_channels = 32,
        kernel_size = 3,
        stride = 2,
        padding = 1
    )
    conv2 = Conv2d(
        in_channels = 32,
        out_channels = 32,
        kernel_size = 3,
        stride = 2,
        padding = 1
    )
    W1, b1, W2, b2 = create_layers(
        in_features,
        out_features,
        hidden_size = hidden_size,
        mode=init_mode,
        seed=seed
    )
    dropout = Dropout()


    train_acc_history = []
    train_loss_history = []
    val_acc_history = []
    val_loss_history = []
    for epoch in range(epochs):

        if epoch == 26:
            opt.lr = 1e-4

        """    Training start    """
        imgs_batches, targets_batches = data_loader(
            train_data,
            in_features,
            out_features,
            batch_size,
            epoch,
            shuffle = shuffle,
            seed = seed,
        )

        if norm:
            imgs_batches = normalize(imgs_batches)

        train_loop = tqdm(
            zip(imgs_batches, targets_batches),
            leave=False
        )

        correct = 0
        total = 0
        batch_losses = []
        for batch_idx, (img_batch, target_batch) in enumerate(train_loop):
            # Прямой проход \ Forward
            c1 = conv1.forward(img_batch)
            act_c1 = relu(c1)
            c2 = conv2.forward(act_c1)
            act_c2 = relu(c2)
            liniar_batch = act_c2.reshape(batch_size, -1)
            out1 = liniar_batch @ W1 + b1
            act1 = relu(out1)
            drop = dropout(act1)
            pred = drop @ W2 + b2

            # Функция потерь \ Loss
            loss = loss_model(pred, target_batch)
            batch_losses.append(loss)

            # Обратный проход \ Backward
            dpred = loss_model.backward()

            dW2 = drop.T @ dpred
            db2 = np.sum(dpred, axis=0, keepdims=True)

            d_drop = dpred @ W2.T

            dact1 = dropout.backward(d_drop)

            dout1 = drelu(act1, dact1)

            dW1 = liniar_batch.T @ dout1
            db1 = np.sum(dout1, axis=0, keepdims=True)

            d_liniar_batch = dout1 @ W1.T
            d_act_c2 = d_liniar_batch.reshape(act_c2.shape)

            d_c2 = drelu(act_c2, d_act_c2)
            d_act_c1 = conv2.backward(d_c2)

            d_c1 = drelu(act_c1, d_act_c1)
            _ = conv1.backward(d_c1)

            # Шаг оптимизации
            params = [
                conv1.w, conv1.b,
                conv2.w, conv2.b,
                W2, b2,
                W1, b1
            ]
            grads = [
                conv1.dw, conv1.db,
                conv2.dw, conv2.db,
                dW2, db2,
                dW1, db1
            ]
            conv1.w, conv1.b, conv2.w, conv2.b, W2, b2, W1, b1 = opt.step(params, grads)

            # Подсчёт метрики
            target_label = to_label(target_batch)
            pred_label = to_label(pred)
            correct += (pred_label == target_label).sum().item()
            total += target_label.size

            if batch_idx % 10 == 0:
                train_loop.set_description(f"Epoch [{epoch+1}/{epochs}], train_loss = {loss:.4f}")

        train_acc = correct / total
        train_loss = np.mean(batch_losses).item()
        train_acc_history.append(train_acc)
        train_loss_history.append(train_loss)

        """    Training end    """

        """    Validation start    """
        imgs_batches, targets_batches = data_loader(
            val_data,
            in_features,
            out_features,
            batch_size,
            epoch,
            shuffle = False,
            seed = seed,
        )

        if norm:
            imgs_batches = normalize(imgs_batches)

        train_loop = zip(imgs_batches, targets_batches)

        correct = 0
        total = 0
        batch_losses = []
        for batch_idx, (img_batch, target_batch) in enumerate(train_loop):
            # Прямой проход \ Forward
            c1 = conv1.forward(img_batch)
            act_c1 = relu(c1)
            c2 = conv2.forward(act_c1)
            act_c2 = relu(c2)
            liniar_batch = act_c2.reshape(batch_size, -1)
            out1 = liniar_batch @ W1 + b1
            act1 = relu(out1)
            drop = dropout(act1)
            pred = drop @ W2 + b2

            # Функция потерь \ Loss
            loss = loss_model(pred, target_batch)
            batch_losses.append(loss)

            # Подсчёт метрики
            target_label = to_label(target_batch)
            pred_label = to_label(pred)
            correct += (pred_label == target_label).sum().item()
            total += target_label.size

        val_acc = correct / total
        val_loss = np.mean(batch_losses).item()
        val_acc_history.append(val_acc)
        val_loss_history.append(val_loss)

        """    Validation end    """

        print(f"Epoch [{epoch+1}/{epochs}], train_loss = {train_loss:.4f}, val_loss = {val_loss:.4f}")

    params = [
        conv1.w, conv1.b,
        conv2.w, conv2.b,
        W2, b2,
        W1, b1
    ]

    print(f"\nПосле {epoch+1} эпох обучения:")
    print(f"\ttrain_loss = {train_loss:.4f}, train_acc = {train_acc:.4f}")
    print(f"\tval_loss   = {val_loss:.4f}, val_acc   = {val_acc:.4f}")

    # Строю графики Loss и Accuracy.
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))

    ax1.plot(train_loss_history, label=f'train')
    ax1.plot(val_loss_history, label=f'val')
    ax1.set_title('Loss')
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)

    ax2.plot(train_acc_history, label=f'train')
    ax2.plot(val_acc_history, label=f'val')
    ax2.set_title('Accuracy')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)

    plt.show()

    return params



In [ ]:
# Создаю набор данных.
imgs, targets = load_mnist_simple()

row = 2
col = 3
idxs = np.random.randint(0, 60000, row*col)
fig, ax = plt.subplots(row, col, figsize=(14, 7))
for i, idx in enumerate(idxs):
    img, one_hot = imgs[idx], targets[idx]
    ax[i//col, i%col].imshow(img, cmap='gray')
    ax[i//col, i%col].set_title(f"{one_hot}")
    ax[i//col, i%col].set_xticks([])
    ax[i//col, i%col].set_yticks([])

plt.show()

In [ ]:
seed = 1961

# Сначала отделяем тест
t_img, test_img, t_target, test_target = train_test_split(
    imgs, targets, test_size=0.5, random_state=seed, stratify=np.argmax(targets, axis=1)
)
# Затем делим оставшееся на train/val
train_img, val_img, train_target, val_target = train_test_split(
    t_img, t_target, test_size=0.1, random_state=seed, stratify=np.argmax(t_target, axis=1)
)

# Данные для обучения
train_data = {
    'train': (train_img, train_target),
    'val': (val_img, val_target),
}

# Функция потерь
loss_model = CrossEntropyLoss()

# Оптимизатор
opt = AdamW()

params = train(
    loss_model,
    opt,
    train_data,
    in_features = 7*7*32,
    hidden_size = 64,
    out_features = 10,
    init_mode = "uniform",
    batch_size = 64,
    epochs = 5,
    shuffle = True,
    norm = True,
    seed = seed
)

In [27]:
import torch
import torch.nn as nn

In [28]:

# Параметры
N, C_in, C_out, H, W = 2, 3, 5, 28, 28
K, S, P = 3, 1, 0

# Данные
np.random.seed(42)
torch.manual_seed(42)

x_np = np.random.randn(N, C_in, H, W).astype(np.float32)
x_torch = torch.from_numpy(x_np).requires_grad_(True)

# NumPy слой
conv_np = Conv2d(C_in, C_out, K, S, P)

# PyTorch слой с теми же весами
conv_torch = nn.Conv2d(C_in, C_out, K, S, P, bias=True)
with torch.no_grad():
    conv_torch.weight.copy_(torch.from_numpy(conv_np.w))
    conv_torch.bias.copy_(torch.from_numpy(conv_np.b.flatten()))

# Forward
out_np = conv_np.forward(x_np)
out_th = conv_torch(x_torch)

print(f"Forward — Max diff: {np.max(np.abs(out_np - out_th.detach().numpy())):.2e}")

# Backward
dout = np.random.randn(*out_np.shape).astype(np.float32)
dx_np = conv_np.backward(dout)

out_th.backward(torch.from_numpy(dout))

print(f"dx — Max diff: {np.max(np.abs(dx_np - x_torch.grad.numpy())):.2e}")
print(f"dw — Max diff: {np.max(np.abs(conv_np.dw - conv_torch.weight.grad.numpy())):.2e}")
print(f"db — Max diff: {np.max(np.abs(conv_np.db.flatten() - conv_torch.bias.grad.numpy())):.2e}")


Forward — Max diff: 8.84e-07
dx — Max diff: 8.49e-07
dw — Max diff: 3.43e-05
db — Max diff: 2.29e-05
